# InsightStory-EDA: E-commerce customer behaviour

**Business questions**
1. Why is customer churn rising?
2. Which customers and products drive profit?
3. Where are we leaking margin?

**Hypotheses**
- H1: Acquisition channel drives repeat purchase and churn.
- H2: A small core of customers drives most revenue and profit.
- H3: Heavy discounting in low-margin categories erodes profit.

Run `insightstory build-db` first (see README). All SQL lives in `sql/`; this notebook only runs it and plots the results.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from insightstory import analysis, charts, insights
from insightstory.config import load_settings
from insightstory.db import connect

cfg = load_settings()
con = connect(cfg, read_only=True)
frames = analysis.run_all(con, cfg)
ins = insights.build(frames, cfg)
list(frames)

## 1. Data overview
Four tables: `customers`, `products`, `orders`, `order_items`. Revenue, cost and profit are derived in the `v_order_lines` view (completed orders only).

In [ ]:
pd.Series(
    {
        t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        for t in ["customers", "products", "orders", "order_items"]
    },
    name="rows",
)

In [ ]:
frames["q01_kpi_summary"]

## 2. Revenue trend (`q06`: LAG, rolling average, running total)

In [ ]:
fig = charts.revenue_trend(frames["q06_monthly_trend"], ins.takeaways["revenue_trend"])
plt.show()

## 3. Margin by category (`q02`)
H3: discounts erode profit in low-margin categories.

In [ ]:
display(frames["q02_category_margin"])
fig = charts.category_margin(frames["q02_category_margin"], ins.takeaways["category_margin"])
plt.show()

## 4. RFM segmentation (`v_rfm`, `q03`)
Scores use `NTILE(5)`. Frequency has many ties at 1 order, so ties are broken by monetary value.

In [ ]:
display(frames["q03_rfm_segments"])
fig = charts.rfm_segments(frames["q03_rfm_segments"], ins.takeaways["rfm_segments"])
plt.show()

## 5. Churn by acquisition channel (`q04`)
H1: channel drives repeat purchase. A customer is churned when the last order is older than the configured number of days.

In [ ]:
display(frames["q04_churn_by_channel"])
fig = charts.churn_by_channel(frames["q04_churn_by_channel"], ins.takeaways["churn_by_channel"])
plt.show()

## 5b. Churn trend over time (`q09`)
Is churn rising? At each month end we count customers whose last order is older than the churn window, among everyone who had ordered by then. Early months are skipped as warm-up, because nobody can be lapsed before one churn window has passed.

In [ ]:
display(frames["q09_churn_trend"].tail(8))
fig = charts.churn_trend(frames["q09_churn_trend"], ins.takeaways["churn_trend"])
plt.show()

## 6. Cohort retention (`q05`)

In [ ]:
fig = charts.cohort_heatmap(frames["q05_cohort_retention"], ins.takeaways["cohort_retention"])
plt.show()

## 7. Profit concentration (`q07`)
H2: a small core of customers drives most profit.

In [ ]:
display(frames["q07_profit_concentration"])
fig = charts.profit_concentration(
    frames["q07_profit_concentration"], ins.takeaways["profit_concentration"]
)
plt.show()

## 8. Distributions and correlations

In [ ]:
rfm = con.execute("SELECT recency_days, frequency, monetary, profit FROM v_rfm").df()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(rfm["recency_days"], bins=30, ax=axes[0])
sns.histplot(rfm["frequency"], bins=range(1, 30), ax=axes[1])
sns.histplot(rfm["monetary"], bins=40, ax=axes[2])
axes[0].set_title("Recency (days since last order)")
axes[1].set_title("Frequency (orders)")
axes[2].set_title("Monetary (revenue per customer, $)")
plt.tight_layout()
plt.show()

In [ ]:
sns.heatmap(rfm.corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlation: recency, frequency, monetary, profit")
plt.show()

## 9. Recommendations (each tied to a query metric)

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(ins.recommendations))

## 10. Query accuracy check
Reconcile SQL KPIs against an independent calculation from the raw tables.

In [ ]:
revenue_sql = float(frames["q01_kpi_summary"].loc[0, "revenue"])
revenue_raw = float(frames["q08_reconciliation"].loc[0, "revenue_raw"])
revenue_rfm = float(frames["q03_rfm_segments"]["revenue"].sum())
assert abs(revenue_sql - revenue_raw) < 0.01, "KPI revenue differs from raw-table revenue"
assert abs(revenue_sql - revenue_rfm) < 1.0, "RFM segment revenue does not reconcile"
print("Reconciliation passed:", revenue_sql, revenue_raw, revenue_rfm)
con.close()